In [3]:
import sys
sys.argv = ['']

!pip install pandas numpy scikit-learn imbalanced-learn joblib -q

In [4]:
from google.colab import files
uploaded = files.upload()


Saving data.csv to data.csv


In [12]:

import os, sys, math, string, argparse, warnings
import numpy as np
import pandas as pd
import joblib

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, classification_report,
    confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score)
from sklearn.preprocessing import StandardScaler, label_binarize
from imblearn.over_sampling import SMOTE

warnings.filterwarnings("ignore")

MODEL_FILE = "rf_model.joblib"
DATA_FILE  = "data.csv"

LEET_MAP = {'@':'a','4':'a','$':'s','5':'s','1':'l','!':'l','0':'o','3':'e','7':'t'}

KEYBOARD_WALKS = [
    "qwerty","asdfg","zxcvb","qazwsx","1qaz","2wsx",
    "12345","123456","1234567","12345678","123456789","abcdef"
]

BREACHED = {
    "password","123456","12345678","qwerty","abc123","monkey","1234567","letmein",
    "trustno1","dragon","baseball","iloveyou","master","sunshine","shadow","123123",
    "654321","superman","qazwsx","michael","football","password1","princess","batman",
    "access","hello","charlie","jessica","daniel","hunter","thomas","welcome","winter",
    "summer","admin","user","guest","root","pass","login","secret","1234","12345",
    "000000","111111","666666","123321"
}

DICT_ROOTS = [
    "password","pass","welcome","login","admin","qwerty","letmein","monkey","dragon",
    "master","shadow","sunshine","princess","football","baseball","batman","superman",
    "iloveyou","michael","jessica","charlie","thomas","hunter","winter","summer",
    "flower","secret","freedom","access","hello"
]

LABELS = {0: "WEAK", 1: "MEDIUM", 2: "STRONG"}

FEATURE_NAMES = [
    "length","num_digits","num_upper","num_lower","num_special",
    "char_variety","dynamic_charset","entropy",
    "keyboard_walk","sequential","repeated","dict_word"
]

def normalize_leet(pw):
    return ''.join(LEET_MAP.get(c, c) for c in pw.lower())

def shannon_entropy(s):
    if not s:
        return 0.0
    freq = {}
    for c in s:
        freq[c] = freq.get(c, 0) + 1
    n = len(s)
    return -sum((v/n) * math.log2(v/n) for v in freq.values())

def get_features(pw):
    norm = normalize_leet(pw)
    variety = (
        (1 if any(c.islower() for c in pw) else 0) +
        (1 if any(c.isupper() for c in pw) else 0) +
        (1 if any(c.isdigit() for c in pw) else 0) +
        (1 if any(c in string.punctuation for c in pw) else 0)
    )
    charset = 0
    if any(c.islower() for c in pw): charset += 26
    if any(c.isupper() for c in pw): charset += 26
    if any(c.isdigit() for c in pw): charset += 10
    if any(c in string.punctuation for c in pw): charset += 32

    walk = 1 if any(w in pw.lower() for w in KEYBOARD_WALKS) else 0

    seq = 0
    for i in range(len(pw) - 2):
        a, b, c = ord(pw[i]), ord(pw[i+1]), ord(pw[i+2])
        if (b-a == 1 and c-b == 1) or (a-b == 1 and b-c == 1):
            seq = 1; break

    rep = 1 if any(pw[i]==pw[i+1]==pw[i+2] for i in range(len(pw)-2)) else 0

    p = pw.lower()
    dict_hit = 1 if p in BREACHED or any(r in p for r in DICT_ROOTS) else 0

    return {
        "length": len(pw),
        "num_digits": sum(c.isdigit() for c in pw),
        "num_upper": sum(c.isupper() for c in pw),
        "num_lower": sum(c.islower() for c in pw),
        "num_special": sum(c in string.punctuation for c in pw),
        "char_variety": variety,
        "dynamic_charset": charset,
        "entropy": round(shannon_entropy(norm), 4),
        "raw_entropy": round(shannon_entropy(pw), 4),
        "normalized": norm,
        "keyboard_walk": walk,
        "sequential": seq,
        "repeated": rep,
        "dict_word": dict_hit
    }

def to_vector(pw):
    f = get_features(pw)
    return [f["length"], f["num_digits"], f["num_upper"], f["num_lower"],
            f["num_special"], f["char_variety"], f["dynamic_charset"], f["entropy"],
            f["keyboard_walk"], f["sequential"], f["repeated"], f["dict_word"]]

def train(data_path, fast=False):



    df = pd.read_csv(data_path, on_bad_lines='skip')
    df.columns = [c.strip().lower() for c in df.columns]
    df = df.dropna(subset=['password','strength']).drop_duplicates(subset=['password'])
    df['password'] = df['password'].astype(str)
    df['strength'] = df['strength'].astype(int)
    df = df[df['strength'].isin([0,1,2])]


    if fast:
        df = df.sample(50000, random_state=42).reset_index(drop=True)




    X = np.array([to_vector(pw) for pw in df['password']], dtype=np.float32)
    y = df['strength'].values

    X_tv, X_test, y_tv, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)
    X_train, X_val, y_train, y_val = train_test_split(X_tv, y_tv, test_size=0.111, stratify=y_tv, random_state=42)



    X_train, y_train = SMOTE(random_state=42).fit_resample(X_train, y_train)


    scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    X_test_sc  = scaler.transform(X_test)

    print("\nTraining models...")
    rf  = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    svm = SVC(kernel='rbf', probability=True, random_state=42)
    lr  = LogisticRegression(max_iter=1000, random_state=42)

    rf.fit(X_train, y_train);
    svm.fit(X_train_sc, y_train);
    lr.fit(X_train_sc, y_train);


    y_test_bin = label_binarize(y_test, classes=[0,1,2])

    print("\nresult")
    print(f"\n{'Model':<24} {'Accuracy':>10} {'Precision':>10} {'Recall':>9} {'F1':>8} {'ROC-AUC':>9}")


    model_list = [
        ("Random Forest",       rf,  X_test,    ),
        ("SVM (RBF)",           svm, X_test_sc, ),
        ("Logistic Regression", lr,  X_test_sc, ),
    ]

    for name, m, Xt in model_list:
        pred  = m.predict(Xt)
        proba = m.predict_proba(Xt)
        acc   = accuracy_score(y_test, pred)
        prec  = precision_score(y_test, pred, average='weighted')
        rec   = recall_score(y_test, pred, average='weighted')
        f1    = f1_score(y_test, pred, average='weighted')
        auc   = roc_auc_score(y_test_bin, proba, multi_class='ovr', average='weighted')
        print(f"{name:<24} {acc:>10.4f} {prec:>10.4f} {rec:>9.4f} {f1:>8.4f} {auc:>9.4f}")

    rf_pred = rf.predict(X_test)
    print("\nDetailed report - Random Forest:")
    print(classification_report(y_test, rf_pred, target_names=['Weak','Medium','Strong']))


    print("Confusion matrices (rows=true, cols=predicted):\n")
    for name, m, Xt in model_list:
        pred = m.predict(Xt)
        cm   = confusion_matrix(y_test, pred)
        print(f"  {name}")
        print(f"  {'':12} {'Pred Weak':>12} {'Pred Medium':>12} {'Pred Strong':>12}")
        for i, row in enumerate(cm):
            print(f"  {'True '+LABELS[i]:<12} {row[0]:>12,} {row[1]:>12,} {row[2]:>12,}")
        fn = cm[0][2]
        print(f"  Critical FN (Weak predicted as Strong): {fn:,} ({100*fn/cm[0].sum():.2f}%)\n")

    rf_proba = rf.predict_proba(X_test)
    print("ROC-AUC per class - Random Forest:")
    for i, cls in enumerate(['Weak','Medium','Strong']):
        auc = roc_auc_score((y_test == i).astype(int), rf_proba[:, i])
        print(f"  {cls}: {auc:.4f}")

    # feature importance
    print("\nFeature importance - Random Forest:")
    pairs = sorted(zip(FEATURE_NAMES, rf.feature_importances_), key=lambda x: -x[1])
    for fname, score in pairs:
        print(f"  {fname:<22} {score:.4f}  ")

    joblib.dump({"rf": rf, "svm": svm, "lr": lr, "scaler": scaler}, MODEL_FILE)



def load_model():
    if os.path.exists(MODEL_FILE):
        bundle = joblib.load(MODEL_FILE)
        return bundle["rf"], bundle["scaler"], True
    return None, None, False

def predict(pw, rf, scaler, model_loaded):
    vec = np.array([to_vector(pw)], dtype=np.float32)
    if model_loaded:
        proba = rf.predict_proba(vec)[0]
        score = int(np.argmax(proba))
        conf  = float(proba[score])
    else:
        f = get_features(pw)
        pts = 0
        if f['length'] >= 12: pts += 2
        elif f['length'] >= 8: pts += 1
        if f['char_variety'] >= 4: pts += 2
        elif f['char_variety'] >= 3: pts += 1
        if f['entropy'] >= 3.0: pts += 2
        elif f['entropy'] >= 2.0: pts += 1
        if f['dict_word']: pts -= 2
        if f['keyboard_walk']: pts -= 1
        if f['sequential']: pts -= 1
        score = 0 if pts <= 1 else (1 if pts <= 3 else 2)
        conf  = 0.75
    return score, conf

def show_result(pw, rf, scaler, model_loaded):
    score, conf = predict(pw, rf, scaler, model_loaded)
    f = get_features(pw)

    print(f"\nPassword : {pw}")
    print(f"Result   : {LABELS[score]}  ({conf*100:.1f}% confidence)")
    print(f"\nFeatures:")
    print(f"  Length             : {f['length']}")
    print(f"  Char variety (0-4) : {f['char_variety']}")
    print(f"  Dynamic charset    : {f['dynamic_charset']}")
    print(f"  Raw entropy        : {f['raw_entropy']}")
    print(f"  Leet-norm entropy  : {f['entropy']}  (normalized: '{f['normalized']}')")
    print(f"  Keyboard walk      : {'YES' if f['keyboard_walk'] else 'No'}")
    print(f"  Sequential pattern : {'YES' if f['sequential'] else 'No'}")
    print(f"  Repeated chars     : {'YES' if f['repeated'] else 'No'}")
    print(f"  Dictionary/breached: {'YES' if f['dict_word'] else 'No'}")

    print(f"\nFeedback:")
    if f['length'] < 8:
        print(f"  - Too short ({f['length']} chars), use at least 12")
    elif f['length'] < 12:
        print(f"  - Acceptable length but aim for 12+")
    else:
        print(f"  - Good length")
    if f['dict_word']:
        print(f"  - Contains a common/breached word")
    if f['keyboard_walk']:
        print(f"  - Keyboard walk detected (e.g. qwerty, 12345)")
    if f['sequential']:
        print(f"  - Sequential characters detected")
    if f['repeated']:
        print(f"  - Repeated characters detected")
    if f['raw_entropy'] - f['entropy'] > 0.1:
        print(f"  - Leet substitution detected: true entropy is {f['entropy']}, not {f['raw_entropy']}")
    if f['char_variety'] < 3:
        print(f"  - Low character variety ({f['char_variety']}/4), mix upper/lower/digits/symbols")
    print()

def show_comparison(rf, scaler, model_loaded):
    passwords = [
        "password", "P@ssw0rd1!", "qwerty123", "L3tm31n!!",
        "BlueSky88", "Summer2024!", "xK#9mP!2qZ", "Tr7$kM!w9Qz#2aB", "abc12345"
    ]

    print("\n=== COMPARISON TABLE ===")
    print("Passwords that pass LUDS rules but are caught by the ML model\n")
    print(f"{'Password':<22} {'Label':>8} {'Conf':>7} {'Entropy':>9} {'Variety':>9} {'Dict':>6} {'Pattern':>9}")
    print("-" * 72)

    for pw in passwords:
        score, conf = predict(pw, rf, scaler, model_loaded)
        f = get_features(pw)
        d = "YES" if f['dict_word'] else "No"
        p = "YES" if (f['keyboard_walk'] or f['sequential']) else "No"
        print(f"{pw:<22} {LABELS[score]:>8} {conf*100:>6.1f}% {f['entropy']:>9} {f['char_variety']:>9} {d:>6} {p:>9}")

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--train",    action="store_true")
    parser.add_argument("--fast",     action="store_true")
    parser.add_argument("--demo",     action="store_true")
    parser.add_argument("--password", type=str)
    parser.add_argument("--data",     type=str, default=DATA_FILE)
    args = parser.parse_args()

    if args.train:
        if not os.path.exists(args.data):
            print(f"Dataset not found: {args.data}")
            sys.exit(1)
        train(args.data, fast=args.fast)
        return

    rf, scaler, model_loaded = load_model()
    if not model_loaded:
        print("No trained model found.")


    if args.password:
        show_result(args.password, rf, scaler, model_loaded)
        return

    if args.demo:
        show_comparison(rf, scaler, model_loaded)
        return


    while True:
        try:
            pw = input("Enter password: ").strip()
        except (KeyboardInterrupt, EOFError):
            break
        if not pw:
            continue
        if pw.lower() in ('quit', 'exit'):
            break
        if pw.lower() == 'demo':
            show_comparison(rf, scaler, model_loaded)
            continue
        show_result(pw, rf, scaler, model_loaded)

import sys
sys.argv = ['']


In [13]:
train("data.csv", fast=True)  # fast=True uses 50k rows, quicker


Training models...

result

Model                      Accuracy  Precision    Recall       F1   ROC-AUC
Random Forest                1.0000     1.0000    1.0000   1.0000    1.0000
SVM (RBF)                    0.9989     0.9989    0.9989   0.9989    1.0000
Logistic Regression          0.9999     0.9999    0.9999   0.9999    1.0000

Detailed report - Random Forest:
              precision    recall  f1-score   support

        Weak       1.00      1.00      1.00      1351
      Medium       1.00      1.00      1.00      7419
      Strong       1.00      1.00      1.00      1230

    accuracy                           1.00     10000
   macro avg       1.00      1.00      1.00     10000
weighted avg       1.00      1.00      1.00     10000

Confusion matrices (rows=true, cols=predicted):

  Random Forest
                  Pred Weak  Pred Medium  Pred Strong
  True WEAK           1,351            0            0
  True MEDIUM             0        7,419            0
  True STRONG            

In [14]:
rf, scaler, model_loaded = load_model()

In [15]:
show_result("WeeDis4W0r$l!@#d", rf, scaler, model_loaded)


Password : WeeDis4W0r$l!@#d
Result   : STRONG  (100.0% confidence)

Features:
  Length             : 16
  Char variety (0-4) : 4
  Dynamic charset    : 94
  Raw entropy        : 3.75
  Leet-norm entropy  : 3.25  (normalized: 'weedisaworslla#d')
  Keyboard walk      : No
  Sequential pattern : No
  Repeated chars     : No
  Dictionary/breached: No

Feedback:
  - Good length
  - Leet substitution detected: true entropy is 3.25, not 3.75

